In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Frontend Simulado
# MAGIC
# MAGIC Este notebook simula uma interface simples do JobFlow AI dentro do Databricks.
# MAGIC
# MAGIC Ele mostra:
# MAGIC
# MAGIC - resumo do usuário demo;
# MAGIC - métricas principais;
# MAGIC - vagas recomendadas;
# MAGIC - vagas salvas;
# MAGIC - aplicações criadas.

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

DEMO_USER_ID = "demo_user_001"

APP_USERS_TABLE = f"{CATALOG}.{SCHEMA}.app_users"
APP_PROFILES_TABLE = f"{CATALOG}.{SCHEMA}.app_profiles"
APP_JOB_POSTINGS_TABLE = f"{CATALOG}.{SCHEMA}.app_job_postings"
APP_SAVED_JOBS_TABLE = f"{CATALOG}.{SCHEMA}.app_saved_jobs"
APP_APPLICATIONS_TABLE = f"{CATALOG}.{SCHEMA}.app_applications"
APP_INTERVIEW_NOTES_TABLE = f"{CATALOG}.{SCHEMA}.app_interview_notes"
MATCH_SCORES_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_match_scores"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Frontend simulado iniciado.")
print(f"Schema ativo: {CATALOG}.{SCHEMA}")
print(f"Usuário demo: {DEMO_USER_ID}")

# COMMAND ----------

def count_rows(table_name, filter_user=False):
    df = spark.table(table_name)

    if filter_user and "user_id" in df.columns:
        df = df.where(F.col("user_id") == DEMO_USER_ID)

    return df.count()


total_jobs = count_rows(APP_JOB_POSTINGS_TABLE)
total_recommended = count_rows(MATCH_SCORES_TABLE, filter_user=True)
total_saved = count_rows(APP_SAVED_JOBS_TABLE, filter_user=True)
total_applications = count_rows(APP_APPLICATIONS_TABLE, filter_user=True)
total_notes = count_rows(APP_INTERVIEW_NOTES_TABLE, filter_user=True)

# COMMAND ----------

displayHTML(f"""
<div style="
    padding: 24px;
    border-radius: 16px;
    background: linear-gradient(135deg, #1f2937, #111827);
    color: white;
    font-family: Arial, sans-serif;
">
    <h1 style="margin-bottom: 4px;">JobFlow AI</h1>
    <p style="font-size: 16px; margin-top: 0;">
        Copiloto de busca, recomendação e acompanhamento de vagas.
    </p>

    <div style="display: flex; gap: 16px; margin-top: 24px; flex-wrap: wrap;">
        <div style="background: #374151; padding: 16px; border-radius: 12px; min-width: 160px;">
            <div style="font-size: 28px; font-weight: bold;">{total_jobs}</div>
            <div>Vagas publicadas</div>
        </div>

        <div style="background: #374151; padding: 16px; border-radius: 12px; min-width: 160px;">
            <div style="font-size: 28px; font-weight: bold;">{total_recommended}</div>
            <div>Vagas ranqueadas</div>
        </div>

        <div style="background: #374151; padding: 16px; border-radius: 12px; min-width: 160px;">
            <div style="font-size: 28px; font-weight: bold;">{total_saved}</div>
            <div>Vagas salvas</div>
        </div>

        <div style="background: #374151; padding: 16px; border-radius: 12px; min-width: 160px;">
            <div style="font-size: 28px; font-weight: bold;">{total_applications}</div>
            <div>Aplicações</div>
        </div>

        <div style="background: #374151; padding: 16px; border-radius: 12px; min-width: 160px;">
            <div style="font-size: 28px; font-weight: bold;">{total_notes}</div>
            <div>Notas de entrevista</div>
        </div>
    </div>
</div>
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Perfil do usuário demo

# COMMAND ----------

display(
    spark.table(APP_USERS_TABLE)
    .where(F.col("user_id") == DEMO_USER_ID)
)

display(
    spark.table(APP_PROFILES_TABLE)
    .where(F.col("user_id") == DEMO_USER_ID)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Top vagas recomendadas

# COMMAND ----------

recommended_jobs_df = (
    spark.table(MATCH_SCORES_TABLE).alias("m")
    .where(F.col("m.user_id") == DEMO_USER_ID)
    .join(
        spark.table(APP_JOB_POSTINGS_TABLE).alias("j"),
        F.col("m.job_id") == F.col("j.job_id"),
        "left"
    )
    .orderBy(F.col("m.match_score").desc())
    .limit(10)
)

display(recommended_jobs_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Vagas salvas pelo usuário

# COMMAND ----------

saved_jobs_df = (
    spark.table(APP_SAVED_JOBS_TABLE).alias("s")
    .where(F.col("s.user_id") == DEMO_USER_ID)
    .join(
        spark.table(APP_JOB_POSTINGS_TABLE).alias("j"),
        F.col("s.job_id") == F.col("j.job_id"),
        "left"
    )
    .orderBy(F.col("s.updated_at").desc())
)

display(saved_jobs_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Aplicações criadas

# COMMAND ----------

applications_df = (
    spark.table(APP_APPLICATIONS_TABLE).alias("a")
    .where(F.col("a.user_id") == DEMO_USER_ID)
    .join(
        spark.table(APP_JOB_POSTINGS_TABLE).alias("j"),
        F.col("a.job_id") == F.col("j.job_id"),
        "left"
    )
    .orderBy(F.col("a.updated_at").desc())
)

display(applications_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Notas de entrevista

# COMMAND ----------

display(
    spark.table(APP_INTERVIEW_NOTES_TABLE)
    .where(F.col("user_id") == DEMO_USER_ID)
    .orderBy(F.col("created_at").desc())
)

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: FRONTEND SIMULADO CONCLUÍDO")
print("=" * 70)
print("O notebook frontend_simulado.py apresenta uma interface simples do JobFlow AI.")
print("Ele mostra métricas, perfil demo, vagas recomendadas, vagas salvas e aplicações.")
print("=" * 70)